# PyTorch: Neural Networks & PINNs

## Overview
PyTorch is a deep learning framework for engineering applications. It provides:
- Automatic differentiation
- Neural network building blocks
- GPU acceleration
- Physics-Informed Neural Networks (PINNs)

## Why PyTorch for Engineering?
- Build surrogate models for CFD
- Implement PINNs for PDE solving
- Accelerate optimization with neural networks

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# 1. Basic Neural Network
print("=== Basic Neural Network ===")

# Define a simple neural network
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )
    
    def forward(self, x):
        return self.net(x)

# Create the network
model = SimpleNN(input_size=2, hidden_size=32, output_size=1)
print(f"Model architecture:")
print(model)

# 2. Training the Network
print("\n=== Training the Network ===")

# Generate training data
np.random.seed(42)
X_train = np.random.randn(1000, 2).astype(np.float32)
y_train = np.sin(X_train[:, 0]) * np.cos(X_train[:, 1]).astype(np.float32)

# Convert to PyTorch tensors
X_train_torch = torch.from_numpy(X_train)
y_train_torch = torch.from_numpy(y_train).unsqueeze(1)

# Define loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
n_epochs = 100
losses = []

for epoch in range(n_epochs):
    # Forward pass
    y_pred = model(X_train_torch)
    loss = criterion(y_pred, y_train_torch)
    
    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.6f}")

# Plot training loss
plt.figure(figsize=(10, 6))
plt.plot(losses, 'b-')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Evaluate the Network
print("\n=== Evaluate the Network ===")

# Generate test data
X_test = np.random.randn(100, 2).astype(np.float32)
y_test = np.sin(X_test[:, 0]) * np.cos(X_test[:, 1]).astype(np.float32)

# Predict
with torch.no_grad():
    y_pred = model(torch.from_numpy(X_test)).numpy()

# Compute error
error = np.abs(y_pred.flatten() - y_test.flatten())
print(f"Test error: mean={error.mean():.6f}, max={error.max():.6f}")

# Plot predictions vs actual
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred.flatten(), alpha=0.5, edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Predictions vs Actual')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Physics-Informed Neural Networks (PINNs)

Solve PDEs using neural networks with physics constraints.

In [ ]:
# 4. PINN: Solve a Simple PDE
print("\n=== PINN: Solve a Simple PDE ===")

# Define the PDE: u_xx + u_yy = 0 (Laplace equation)
# with boundary conditions

class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.net(x)

# Create the PINN
pinn = PINN()
optimizer = torch.optim.Adam(pinn.parameters(), lr=0.001)

# Generate training points
np.random.seed(42)
n_points = 1000
X_collocation = torch.rand(n_points, 2) * 2 - 1  # [-1, 1] x [-1, 1]
X_collocation.requires_grad = True

# Boundary conditions
n_bc = 100
X_bc_left = torch.cat([torch.zeros(n_bc, 1), torch.rand(n_bc, 1) * 2 - 1], dim=1)
X_bc_right = torch.cat([torch.ones(n_bc, 1), torch.rand(n_bc, 1) * 2 - 1], dim=1)
X_bc_bottom = torch.cat([torch.rand(n_bc, 1) * 2 - 1, torch.zeros(n_bc, 1)], dim=1)
X_bc_top = torch.cat([torch.rand(n_bc, 1) * 2 - 1, torch.ones(n_bc, 1)], dim=1)

# Training loop
n_epochs = 1000
losses = []

for epoch in range(n_epochs):
    # Collocation points
    u = pinn(X_collocation)
    
    # Compute derivatives using automatic differentiation
    u_x = torch.autograd.grad(u, X_collocation, create_graph=True, retain_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, X_collocation, create_graph=True, retain_graph=True)[0]
    
    u_y = torch.autograd.grad(u, X_collocation, create_graph=True, retain_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, X_collocation, create_graph=True, retain_graph=True)[0]
    
    # PDE residual: u_xx + u_yy = 0
    pde_residual = u_xx[:, 0] + u_yy[:, 1]
    loss_pde = torch.mean(pde_residual ** 2)
    
    # Boundary conditions
    u_bc_left = pinn(X_bc_left)
    u_bc_right = pinn(X_bc_right)
    u_bc_bottom = pinn(X_bc_bottom)
    u_bc_top = pinn(X_bc_top)
    
    # Simple BC: u = 0 on all boundaries
    loss_bc = (torch.mean(u_bc_left ** 2) + 
               torch.mean(u_bc_right ** 2) + 
               torch.mean(u_bc_bottom ** 2) + 
               torch.mean(u_bc_top ** 2))
    
    # Total loss
    loss = loss_pde + 0.1 * loss_bc
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.6f}")

# Plot training loss
plt.figure(figsize=(10, 6))
plt.plot(losses, 'b-')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('PINN Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercises

1. Train a neural network to approximate a complex function.
2. Implement a PINN for the heat equation.
3. Compare PINN vs traditional numerical methods for a PDE.